<a href="https://colab.research.google.com/github/sandesh-py/ML2/blob/main/Program_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import math

# --------------------------------------------------
# PLAY TENNIS DATASET
# --------------------------------------------------

data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],

    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],

    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],

    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],

    'Play': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)


# --------------------------------------------------
# FOIL INFORMATION GAIN
# --------------------------------------------------

def foil_gain(p0, n0, p1, n1):

    if p1 == 0:
        return -999

    before = p0 * math.log2(p0 / (p0 + n0))

    after = p1 * math.log2(p1 / (p1 + n1))

    return after - before


# --------------------------------------------------
# FIND BEST CONDITION
# --------------------------------------------------

def find_best_condition(data):

    positive = data[data['Play'] == 'Yes']
    negative = data[data['Play'] == 'No']

    p0 = len(positive)
    n0 = len(negative)

    best_condition = None
    best_gain = -999

    for column in data.columns:

        if column == 'Play':
            continue

        for value in data[column].unique():

            positive_subset = positive[
                positive[column] == value
            ]

            negative_subset = negative[
                negative[column] == value
            ]

            p1 = len(positive_subset)
            n1 = len(negative_subset)

            gain = foil_gain(
                p0, n0,
                p1, n1
            )

            if gain > best_gain:
                best_gain = gain
                best_condition = (column, value)

    return best_condition, best_gain


# --------------------------------------------------
# FOIL ALGORITHM
# --------------------------------------------------

def foil(data):

    remaining = data.copy()

    rules = []

    while len(
        remaining[remaining['Play'] == 'Yes']
    ) > 0:

        condition, gain = find_best_condition(
            remaining
        )

        if condition is None:
            break

        column, value = condition

        rule_data = remaining[
            remaining[column] == value
        ]

        positive_count = len(
            rule_data[rule_data['Play'] == 'Yes']
        )

        negative_count = len(
            rule_data[rule_data['Play'] == 'No']
        )

        # Accept condition if it covers more
        # positive examples than negative examples
        if positive_count > negative_count:

            rule = (
                f"IF {column} = {value} "
                f"THEN Play = Yes"
            )

            rules.append(rule)

            # Remove positive examples covered by rule
            remaining = remaining[
                ~(
                    (remaining[column] == value) &
                    (remaining['Play'] == 'Yes')
                )
            ]

        else:
            break

    return rules


# --------------------------------------------------
# DISPLAY DATASET
# --------------------------------------------------

print("\n================ DATASET ================\n")
print(df)


# --------------------------------------------------
# DISPLAY FOIL RULES
# --------------------------------------------------

print("\n================ FOIL RULES ================\n")

rules = foil(df)

for i, rule in enumerate(rules, 1):
    print(f"Rule {i}: {rule}")


================ DATASET ================

     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

================ FOIL RULES ================

Rule 1: IF Outlook = Overcast THEN Play = Yes
Rule 2: IF Temperature = Cool THEN Play = Yes
Rule 3: IF Humidity = Normal THEN Play = Yes
